In [2]:
import sqlite3
import pandas as pd
import os



os.makedirs('chess_db', exist_ok=True)
    
games_url = "https://drive.google.com/uc?export=download&id=1eR3NZtwIC6ECN3vhtrynqmx8okG0twA7"
players_url = "https://drive.google.com/uc?export=download&id=1wCSAkGagMzWiToedLC3ZGo_lGf_laF-k"
    
conn = sqlite3.connect('chess_db/games.db')
conn.execute('PRAGMA foreign_keys = ON')

df1 = pd.read_csv(games_url)
df2 = pd.read_csv(players_url)
    
    

In [3]:
df1 = df1.drop_duplicates(subset = "game_id")
df2 = df2.drop_duplicates(subset = "username")

df1.to_sql('games', conn, if_exists='replace', index=False)
df2.to_sql('players', conn, if_exists='replace', index=False)


215

In [4]:

openings_cols = ['opening_code', 'opening_fullname', 'opening_shortname', 
                 'opening_moves', 'opening_response', 'opening_variation']
df_openings = df1[openings_cols].drop_duplicates('opening_code')
df_openings.to_sql('openings', conn, if_exists='replace', index=False)
openings_cols_new = ['opening_fullname', 'opening_shortname', 
                 'opening_moves', 'opening_response', 'opening_variation']
df_games = df1.drop(columns=openings_cols_new)
df_games.to_sql('games', conn, if_exists='replace', index=False)



20058

In [5]:
conn.execute("CREATE INDEX IF NOT EXISTS idx_games_white_id ON games(white_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_games_black_id ON games(black_id)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_games_opening_code ON games(opening_code)")

In [6]:
explain = conn.execute("""
    EXPLAIN QUERY PLAN 
    SELECT * FROM games WHERE white_id = 'test'
""").fetchall()
print(explain) 

[(3, 0, 61, 'SEARCH games USING INDEX idx_games_white_id (white_id=?)')]


In [7]:
explain = conn.execute("""
    EXPLAIN QUERY PLAN 
    SELECT * FROM games WHERE black_id = 'test'
""").fetchall()
print(explain) 

[(3, 0, 61, 'SEARCH games USING INDEX idx_games_black_id (black_id=?)')]


In [8]:
explain = conn.execute("""
    EXPLAIN QUERY PLAN 
    SELECT * FROM games WHERE opening_code = 'test'
""").fetchall()
print(explain) 

[(3, 0, 61, 'SEARCH games USING INDEX idx_games_opening_code (opening_code=?)')]


In [9]:
conn = sqlite3.connect('chess_db/games.db')

table= conn.execute(
    """SELECT opening_code ,COUNT(*) AS COUNT,
    ROUND(100.0 * SUM(CASE WHEN winner = 'Draw' THEN 1 ELSE 0 END) / COUNT(*), 2) AS draw_rate
    FROM games
    GROUP BY opening_code
    HAVING COUNT(*) >= 20
    ORDER BY draw_rate DESC
    LIMIT 5
    """
).fetchall()
print(table)

[('D43', 33, 21.21), ('A28', 24, 20.83), ('A07', 33, 18.18), ('A08', 39, 15.38), ('D50', 20, 15.0)]


In [10]:
table = conn.execute(
    """SELECT COALESCE(b.player_id , w.player_id) as player_id,
    COALESCE(b.black_wins , 0)AS black_wins,
    COALESCE(w.white_wins , 0)AS white_wins
    FROM (
        SELECT white_id AS player_id  , COUNT(*) AS white_wins
        FROM games
        WHERE winner = "White"
        GROUP BY white_id
    ) w FULL JOIN(
        SELECT black_id AS player_id  , COUNT(*) AS black_wins
        FROM games
        WHERE winner = "Black"
        GROUP BY black_id) b
        ON w.player_id = b.player_id
        WHERE COALESCE(b.black_wins , 0) > COALESCE(w.white_wins , 0)
        ORDER BY black_wins DESC

    """
).fetchall()
print(table)

[('taranga', 38, 34), ('ducksandcats', 29, 14), ('vladimir-kramnik-1', 28, 22), ('chesscarl', 27, 18), ('docboss', 25, 1), ('king5891', 22, 20), ('smilsydov', 21, 15), ('doraemon61', 20, 18), ('artem555', 18, 14), ('cape217', 18, 2), ('dpoty', 18, 8), ('laode_syahril', 18, 16), ('saviter', 18, 14), ('sjeiben', 18, 9), ('smartduckduckcow', 18, 3), ('amanan', 17, 9), ('s-a', 17, 12), ('unrim', 17, 16), ('youralterego', 17, 14), ('lbringer', 16, 14), ('pavelsps', 16, 13), ('rodrigowski19', 16, 12), ('bigbrother101', 15, 11), ('divu7', 15, 8), ('jakubsagitario17', 15, 8), ('juncinmestrella', 15, 11), ('superhero098', 15, 6), ('thatguythatistr', 15, 11), ('aran09', 14, 4), ('chesskhovtsev', 14, 13), ('mahmoud_safyan', 14, 11), ('mazpharaoh', 14, 11), ('moistvonlipwig', 14, 4), ('bear789', 13, 12), ('cro_chess_fan', 13, 10), ('dimitris22', 13, 10), ('dkattir', 13, 9), ('jdbarger', 13, 8), ('jeffersondarcy1', 13, 7), ('loukas435', 13, 9), ('piley159', 13, 8), ('dumbluck', 13, 0), ('basakcesur

In [11]:
conn = sqlite3.connect('chess_db/games.db')

table= conn.execute(
    """SELECT victory_status,opening_code,COUNT(*) AS count 
    FROM games
    GROUP BY victory_status,opening_code
    ORDER BY victory_status,count DESC
    """
).fetchall()
print(table)

[('Draw', 'A00', 39), ('Draw', 'C00', 37), ('Draw', 'D00', 36), ('Draw', 'C50', 29), ('Draw', 'C41', 27), ('Draw', 'A04', 27), ('Draw', 'B01', 25), ('Draw', 'B00', 25), ('Draw', 'C20', 20), ('Draw', 'B20', 20), ('Draw', 'D02', 19), ('Draw', 'B07', 18), ('Draw', 'A40', 17), ('Draw', 'C55', 15), ('Draw', 'A45', 15), ('Draw', 'C45', 12), ('Draw', 'C40', 12), ('Draw', 'C02', 12), ('Draw', 'B50', 12), ('Draw', 'C46', 11), ('Draw', 'C42', 11), ('Draw', 'B23', 11), ('Draw', 'B40', 10), ('Draw', 'B21', 10), ('Draw', 'A03', 10), ('Draw', 'C01', 9), ('Draw', 'B10', 9), ('Draw', 'B02', 9), ('Draw', 'D20', 8), ('Draw', 'D10', 8), ('Draw', 'C23', 8), ('Draw', 'B32', 8), ('Draw', 'B13', 8), ('Draw', 'A06', 8), ('Draw', 'A01', 8), ('Draw', 'D43', 7), ('Draw', 'B12', 7), ('Draw', 'B06', 7), ('Draw', 'A43', 7), ('Draw', 'C62', 6), ('Draw', 'C44', 6), ('Draw', 'B30', 6), ('Draw', 'A46', 6), ('Draw', 'A07', 6), ('Draw', 'D01', 5), ('Draw', 'C70', 5), ('Draw', 'C68', 5), ('Draw', 'C24', 5), ('Draw', 'C10'

In [12]:
table= conn.execute(
    """SELECT o.opening_shortname ,COUNT(*) AS count , 
    ROUND(SUM(turns)* 1.0 /COUNT(*) , 2) AS avg
    FROM games g INNER JOIN openings o 
    ON g.opening_code = o.opening_code
    GROUP BY o.opening_shortname
    HAVING COUNT(*) >= 20
    ORDER BY avg DESC
    LIMIT 3
    """
).fetchall()
print(table)

[("Queen's Indian Defense", 50, 70.86), ("King's Indian Defense", 136, 70.79), ("Queen's Pawn", 75, 68.03)]


In [13]:
conn = sqlite3.connect('chess_db/games.db')
table=conn.execute(
    """WITH all_players AS (
        SELECT white_id AS player_id , game_id , turns
        FROM games UNION ALL 
        SELECT black_id AS player_id , game_id , turns 
        FROM games
    )SELECT player_id , game_id , turns ,
    RANK() OVER (PARTITION BY player_id ORDER BY turns DESC) AS rank_turns
    FROM all_players
    ORDER BY player_id , rank_turns
    limit 20
    """
).fetchall()
print(table)

[('--jim--', 10139, 78, 1), ('-0olo0-', 12540, 39, 1), ('-l-_jedi_knight_-l-', 9799, 138, 1), ('-l-_jedi_knight_-l-', 9791, 112, 2), ('-l-_jedi_knight_-l-', 9798, 99, 3), ('-l-_jedi_knight_-l-', 9789, 93, 4), ('-l-_jedi_knight_-l-', 9790, 92, 5), ('-l-_jedi_knight_-l-', 9796, 88, 6), ('-l-_jedi_knight_-l-', 9792, 74, 7), ('-l-_jedi_knight_-l-', 9795, 73, 8), ('-l-_jedi_knight_-l-', 9794, 39, 9), ('-l-_jedi_knight_-l-', 9797, 36, 10), ('-l-_jedi_knight_-l-', 9788, 35, 11), ('-l-_jedi_knight_-l-', 9793, 28, 12), ('-mati-', 10031, 3, 1), ('-pavel-', 10054, 43, 1), ('1063314', 11091, 141, 1), ('1063314', 11090, 56, 2), ('1111112222', 9036, 122, 1), ('1111112222', 1533, 80, 2)]


In [14]:
columns = ['player_id', 'game_id', 'turns', 'rank_turns']
df = pd.DataFrame(table, columns=columns)
os.makedirs('data/processed', exist_ok=True)
df.to_csv('data/processed/game_ranks.csv', index=False)

In [15]:
table = conn.execute(
    """SELECT game_id,
    ABS(white_rating - black_rating) AS rating_diff,
    turns,
    rated,
    o.opening_shortname AS opening_shortname,
    (
        SELECT COUNT(*) FROM games g2 
        WHERE(g2.white_id = g.white_id  OR g2.black_id  = g.white_id) AND
        (g2.game_id < g.game_id)
    ) AS white_experience ,
    winner
    FROM games g 
    JOIN openings o 
    ON g.opening_code = o.opening_code
    """
)
columns = ['game_id', 'rating_diff', 'turns', 'rated', 'opening_shortname', 'white_experience', 'winner']
df = pd.DataFrame(table.fetchall(), columns=columns)
os.makedirs('data/processed', exist_ok=True)
df.to_csv('data/processed/features.csv', index=False)